In [ ]:
pip install --upgrade --quiet google-cloud-aiplatform[agent_engines,adk] googlemaps requests

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 54.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.5 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.


In [ ]:
import os
import json
import requests
from typing import Tuple, Dict, Any, Optional, Sequence, Callable

# Import Google ADK and Vertex AI Reasoning Engine
import vertexai
from google.adk.agents import Agent
from vertexai.agent_engines import AdkApp

In [ ]:
# --- Configuration ---
PROJECT_ID: str = os.getenv("GOOGLE_CLOUD_PROJECT_ID", "qwiklabs-gcp-04-70cfc463a7f7")
LOCATION: str = os.getenv("GOOGLE_CLOUD_REGION", "us-central1") # e.g., "us-central1"
GOOGLE_MAPS_API_KEY: str = os.getenv("GOOGLE_MAPS_API_KEY", "")

# User-Agent for NWS API requests
# The NWS API requires a User-Agent header. This can be any string identifying your application.
NWS_USER_AGENT: str = "MyWeatherApp/1.0"

# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION)
print(f"Vertex AI Initialized for Project: {PROJECT_ID}, Location: {LOCATION}")

Vertex AI Initialized for Project: qwiklabs-gcp-04-70cfc463a7f7, Location: us-central1


In [ ]:
# --- Tool Definitions (Python functions) ---

def get_lat_long_from_place(place_name: str) -> Optional[Dict[str, float]]:
    """
    Converts a human-readable place name into its latitude and longitude coordinates
    using the Google Maps Geocoding API.

    Args:
        place_name: The name of the place (e.g., "New York City", "London").

    Returns:
        A dictionary containing 'latitude' and 'longitude' as floats, or None if
        geocoding fails.
    """
    if not GOOGLE_MAPS_API_KEY or GOOGLE_MAPS_API_KEY == "YOUR_GOOGLE_MAPS_API_KEY":
        print("Google Maps API Key is not set. Cannot perform geocoding.")
        return None

    geocode_url: str = "https://maps.googleapis.com/maps/api/geocode/json"
    params: Dict[str, str] = {
        "address": place_name,
        "key": GOOGLE_MAPS_API_KEY
    }
    try:
        response = requests.get(geocode_url, params=params, timeout=5)
        response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
        data: Dict[str, Any] = response.json()

        if data["status"] == "OK" and data["results"]:
            location = data["results"][0]["geometry"]["location"]
            return {"latitude": location["lat"], "longitude": location["lng"]}
        else:
            print(f"Geocoding failed for '{place_name}': {data.get('error_message', data['status'])}")
            return None
    except requests.exceptions.RequestException as e:
        print(f"Error during geocoding API request: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"Error decoding geocoding API response: {e}")
        return None

In [ ]:
def get_nws_weather_forecast(latitude: float, longitude: float) -> Optional[Dict[str, Any]]:
    """
    Retrieves the current weather forecast for a given latitude and longitude
    using the National Weather Service (NWS) API.

    This is a two-step process:
    1. Get the forecast office and grid points from the latitude/longitude.
    2. Use the forecast URL from the first step to get the detailed forecast.

    Args:
        latitude: The latitude coordinate.
        longitude: The longitude coordinate.

    Returns:
        A dictionary containing the weather forecast data, or None if the retrieval fails.
    """
    base_url: str = "https://api.weather.gov"
    points_url: str = f"{base_url}/points/{latitude},{longitude}"

    headers: Dict[str, str] = {"User-Agent": NWS_USER_AGENT}

    try:
        # Step 1: Get forecast office and grid points
        response_points = requests.get(points_url, headers=headers, timeout=5)
        response_points.raise_for_status()
        points_data: Dict[str, Any] = response_points.json()

        forecast_url: Optional[str] = points_data["properties"].get("forecast")
        if not forecast_url:
            print(f"Could not find forecast URL for {latitude},{longitude}.")
            return None

        # Step 2: Get the detailed forecast
        response_forecast = requests.get(forecast_url, headers=headers, timeout=5)
        response_forecast.raise_for_status()
        forecast_data: Dict[str, Any] = response_forecast.json()

        return forecast_data

    except requests.exceptions.RequestException as e:
        print(f"Error during NWS API request: {e}")
        return None
    except json.JSONDecodeError as e:
        print(f"Error decoding NWS API response: {e}")
        return None
    except KeyError as e:
        print(f"Missing key in NWS API response: {e}")
        return None

In [ ]:
def get_weather_summary(place_name: str) -> str:
    """
    Provides a weather summary or alert for a given place name.
    This function internally uses get_lat_long_from_place and get_nws_weather_forecast.

    Args:
        place_name: The name of the location for which to get the weather.

    Returns:
        A string containing the weather summary or an error message.
    """
    location_data: Optional[Dict[str, float]] = get_lat_long_from_place(place_name)
    if not location_data:
        return f"Could not find coordinates for {place_name}. Please try again with a different location."

    latitude: float = location_data["latitude"]
    longitude: float = location_data["longitude"]

    weather_data: Optional[Dict[str, Any]] = get_nws_weather_forecast(latitude, longitude)
    if not weather_data:
        return f"Could not retrieve weather data for {place_name}."

    properties: Dict[str, Any] = weather_data.get("properties", {})
    forecast_periods: Optional[list] = properties.get("periods")

    if forecast_periods:
        current_or_next_period: Dict[str, Any] = forecast_periods[0] # Get the current or next forecast period
        name: str = current_or_next_period.get("name", "N/A")
        temperature: str = str(current_or_next_period.get("temperature", "N/A"))
        temperature_unit: str = current_or_next_period.get("temperatureUnit", "")
        forecast_text: str = current_or_next_period.get("detailedForecast", current_or_next_period.get("shortForecast", "No detailed forecast available."))

        summary: str = (
            f"**Weather for {place_name} ({name} Period):**\n"
            f"- Temperature: {temperature}°{temperature_unit}\n"
            f"- Conditions: {forecast_text}\n"
        )
        return summary
    else:
        return f"No forecast periods found for {place_name}."

In [ ]:
# --- Create the Agent ---

al_agent: Agent = Agent(
    model="gemini-2.0-flash", # Using a newer, recommended model version like "gemini-1.5-pro-001"
    name="Al",
    description="A weather agent that provides real-time weather summaries for user-specified locations.",
    instruction=(
        "You are Al, a helpful AI assistant specialized in providing weather information. "
        "Your primary goal is to fetch and summarize real-time weather for any city in the USA provided by the user. "
        "Use your available tools to achieve this. "
        "If the user asks for weather, always try to use the 'get_weather_summary' tool. "
        "If the user says 'bye' or indicates they are finished, provide a polite closing response."
    ),
    tools=[
        get_lat_long_from_place,
        get_nws_weather_forecast,
        get_weather_summary,
    ], # Pass the raw callable functions here
)

weather_app: AdkApp = AdkApp(
    agent=al_agent,
)
print("Agent 'Al' and Weather App created.")

Agent 'Al' and Weather App created.


In [ ]:
# --- Helper function to process agent events ---
def process_agent_event(event: Dict[str, Any]) -> None:
    """
    Processes a single event from the agent's async_stream_query, assuming
    it is a dictionary and extracting information from its 'content.parts' structure.
    """
    # Debug print the full event for examination if needed
    # print(f"DEBUG: Processing event: {json.dumps(event, indent=2)}")

    content = event.get('content')
    if not content or not isinstance(content, dict):
        # print("DEBUG: Event has no 'content' or 'content' is not a dictionary.")
        return

    parts = content.get('parts')
    if not parts or not isinstance(parts, list) or not parts:
        # print("DEBUG: 'content' has no 'parts' or 'parts' is not a non-empty list.")
        return

    # Process each part in the 'parts' list
    for part in parts:
        if not isinstance(part, dict):
            # print("DEBUG: Part is not a dictionary.")
            continue

        # Check for agent's direct text response
        if 'text' in part:
            print(part['text'])

        # Check for tool calls
        if 'function_call' in part:
            function_call_data = part['function_call']
            if isinstance(function_call_data, dict):
                function_name = function_call_data.get('name')
                function_args = function_call_data.get('args')
                # print(f"DEBUG: Agent called tool: {function_name} with args: {function_args}")

        # Check for tool responses
        if 'function_response' in part:
            function_response_data = part['function_response']
            if isinstance(function_response_data, dict) and 'response' in function_response_data:
                response_result = function_response_data['response'].get('result')
                # print(f"DEBUG: Tool response: {response_result}")

In [ ]:
# --- User Session and Interaction ---

async def run_weather_session() -> None:
    """
    Creates a user session and interacts with the weather agent Al.
    Processes input, displays agent responses in Markdown, and handles
    session termination.
    """
    print("\n--- Starting Weather Agent Session ---")
    print("Type 'bye' or 'exit' to end the session.")

    user_id: str = "test_user_123"

    while True:
        user_input: str = input("\nYou: ")
        if user_input.lower() in ["bye", "exit", "quit", "goodbye"]:
            print("\nAl: Goodbye! Have a great day!")
            break

        print("Al:")
        try:
            async for event in weather_app.async_stream_query(
                user_id=user_id,
                message=user_input,
            ):
                # The explicit DEBUG prints for type and content are now less necessary
                # as process_agent_event is tailored to the observed structure,
                # but kept for extreme debugging if needed.
                # print(f"DEBUG: Type of event yielded by async_stream_query: {type(event)}")
                # if isinstance(event, dict):
                #     print(f"DEBUG: Event content (dict): {json.dumps(event, indent=2)}")
                # else:
                #     print(f"DEBUG: Event content (object): {event}") # Should not happen based on current errors

                if isinstance(event, dict): # Ensure it's a dict before passing
                    process_agent_event(event)
                else:
                    print(f"ERROR: Received non-dictionary event: {event}. Cannot process.")


        except Exception as e:
            print(f"An error occurred during agent interaction: {e}")

# To run the async function in a Jupyter notebook
import asyncio
# asyncio.run(run_weather_session())

In [ ]:
# --- Test Code for Multiple Cities in the USA ---

async def test_agent_multiple_cities() -> None:
    """
    Tests the 'Al' agent with a predefined list of US cities and prints
    their weather summaries.
    """
    print("\n--- Testing Agent Al with Multiple US Cities ---")

    cities_to_test: list[str] = [
        "New York City, NY",
        "Los Angeles, CA",
        "Chicago, IL",
        "Houston, TX",
        "Denver, CO"
    ]

    for city in cities_to_test:
        print(f"\n--- Querying weather for {city} ---")
        user_input: str = f"What is the weather like in {city}?"
        print(f"You: {user_input}")
        print("Al:")
        try:
            async for event in weather_app.async_stream_query(
                user_id="test_runner",
                message=user_input,
            ):
                # print(f"DEBUG: Type of event yielded by async_stream_query: {type(event)}")
                # if isinstance(event, dict):
                #     print(f"DEBUG: Event content (dict): {json.dumps(event, indent=2)}")
                # else:
                #     print(f"DEBUG: Event content (object): {event}")

                if isinstance(event, dict): # Ensure it's a dict before passing
                    process_agent_event(event)
                else:
                    print(f"ERROR: Received non-dictionary event: {event}. Cannot process.")
        except Exception as e:
            print(f"An error occurred during agent interaction for {city}: {e}")
        print("-" * 40)

# Run the test cases
await test_agent_multiple_cities()


--- Testing Agent Al with Multiple US Cities ---

--- Querying weather for New York City, NY ---
You: What is the weather like in New York City, NY?
Al:
OK. The weather in New York City, NY today is rain and patchy fog. It is cloudy with a high near 42°F, with temperatures falling to around 39°F in the afternoon. There is a northeast wind of 3 to 10 mph. The chance of precipitation is 100%, with new rainfall amounts between a quarter and half of an inch possible.

----------------------------------------

--- Querying weather for Los Angeles, CA ---
You: What is the weather like in Los Angeles, CA?
Al:
The weather in Los Angeles, CA is sunny with a high near 75°F. The wind is from the north at 10 to 15 mph.

----------------------------------------

--- Querying weather for Chicago, IL ---
You: What is the weather like in Chicago, IL?
Al:
OK. The weather in Chicago, IL today is cloudy with a high near 39°F. There is widespread fog and a chance of rain showers before noon, then areas o